# Profilowanie jakościowe - Milestone 1

Cel: sprawdzenie schematu i jakości danych w warstwie bronze za pomocą zapytań SQL w DuckDB.

Dane: pliki Parquet z `data/bronze/` utworzone w notebookach 02. Notebook tylko czyta dane, niczego nie zapisuje.

## 1. Połączenie z DuckDB i widoki

`duckdb.connect()` bez ścieżki tworzy bazę w pamięci, która istnieje tylko na czas działania notebooka.

Na każdy plik Parquet zakładamy widok (`CREATE VIEW`). Widok to zapisane zapytanie, a nie kopia danych: DuckDB czyta plik dopiero przy zapytaniu i tylko te kolumny, których zapytanie potrzebuje. Dzięki temu nie wczytujemy 9 mln wierszy do RAM.

In [1]:
from pathlib import Path
import duckdb

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
BRONZE_DIR = ROOT / 'data' / 'bronze'

TABLES = ['pomiary_2024', 'stacje', 'stanowiska', 'openmeteo_2024']

con = duckdb.connect()

for table in TABLES:
    path = BRONZE_DIR / f'{table}.parquet'
    con.sql(
        f"CREATE OR REPLACE VIEW {table} AS "
        f"SELECT * FROM read_parquet('{path.as_posix()}')"
    )
    print(f'{table:16} {path.stat().st_size / 1_000_000:6.1f} MB')

print(con.sql('SHOW TABLES'))

pomiary_2024       19.2 MB
stacje              0.1 MB
stanowiska          0.1 MB
openmeteo_2024      7.6 MB
┌────────────────┐
│      name      │
│    varchar     │
├────────────────┤
│ openmeteo_2024 │
│ pomiary_2024   │
│ stacje         │
│ stanowiska     │
└────────────────┘



## 2. Schemat tabel

`DESCRIBE SELECT * FROM ...` zwraca nazwy i typy kolumn. DuckDB odczytuje je z metadanych zapisanych w pliku Parquet, bez skanowania danych.

Kolumny `null` i `key` w wyniku pokazują `YES` i `NULL` dla każdej kolumny, bo Parquet nie przechowuje kluczy ani ograniczeń `NOT NULL`. Klucze opisujemy więc pod każdą tabelą, a ich poprawność sprawdzamy w profilowaniu.

### 2.1. `pomiary_2024`

1 wiersz = 1 pomiar jednego stanowiska w jednym okresie (godzina albo doba).

In [2]:
print(con.sql('DESCRIBE SELECT * FROM pomiary_2024'))

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stanowiska   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ kod_stacji       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wskaznik         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ czas_usredniania │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ jednostka        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_czas        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ wartosc          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ plik_zrodlowy    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



| Kolumna | Znaczenie |
|---|---|
| `kod_stanowiska` | kod przyrządu pomiarowego: stacja, wskaźnik i czas uśredniania, np. `DsWrocAlWisn-PM10-1g` |
| `kod_stacji` | kod stacji, na której stoi przyrząd |
| `wskaznik` | mierzona substancja, np. `PM10`, `PM2.5`, `NO2`, `O3`, `BaP(PM10)` (benzo(a)piren w pyle PM10) |
| `czas_usredniania` | `1g` - wartość godzinowa, `24g` - wartość dobowa |
| `jednostka` | `ug/m3` dla większości wskaźników, `mg/m3` dla CO, `ng/m3` dla śladowych składników pyłu (np. BaP, As, Cd, Ni) i rtęci |
| `data_czas` | znacznik czasu pomiaru z pliku GIOŚ, bez strefy czasowej |
| `wartosc` | zmierzone stężenie; `NULL` oznacza brak pomiaru w tym okresie |
| `plik_zrodlowy` | plik XLSX z warstwy RAW, z którego pochodzi wiersz |

Klucz: `(kod_stanowiska, data_czas)` - jedno stanowisko powinno mieć jeden pomiar w danym okresie. Czy tak jest, sprawdzamy w profilowaniu.

Jednostki różnią się między wskaźnikami, więc statystyki (min, max, średnie) liczymy zawsze osobno dla każdego wskaźnika.

### 2.2. `stacje`

1 wiersz = 1 stacja z metadanych GIOŚ. Arkusz obejmuje wszystkie stacje w historii, także zamknięte przed 2024 r.

In [3]:
print(con.sql('DESCRIBE SELECT * FROM stacje'))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stacji         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ kod_miedzynarodowy │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ nazwa_stacji       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ stary_kod_stacji   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_uruchomienia  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ data_zamkniecia    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ typ_stacji         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ typ_obszaru        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rodzaj_stacji      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │

| Kolumna | Znaczenie |
|---|---|
| `kod_stacji` | kod stacji GIOŚ, np. `DsWrocAlWisn` (województwo, miejscowość, ulica) |
| `kod_miedzynarodowy` | kod stacji w europejskiej bazie jakości powietrza, np. `PL0810A`; nie każda stacja go ma |
| `nazwa_stacji` | nazwa opisowa |
| `stary_kod_stacji` | poprzedni kod, jeśli stacja go zmieniła |
| `data_uruchomienia`, `data_zamkniecia` | okres działania; pusta data zamknięcia - stacja działa |
| `typ_stacji` | czego wpływ ma mierzyć: `tło`, `komunikacyjna` (ruch drogowy), `przemysłowa` |
| `typ_obszaru` | `miejski`, `podmiejski`, `pozamiejski` |
| `rodzaj_stacji` | rodzaj obiektu, np. `kontenerowa stacjonarna`, `mobilna`, `w budynku` |
| `wojewodztwo`, `miejscowosc`, `adres` | lokalizacja administracyjna |
| `szerokosc`, `dlugosc` | współrzędne WGS84 |

Klucz: `kod_stacji`.

### 2.3. `stanowiska`

1 wiersz = 1 stanowisko, czyli jeden przyrząd mierzący jeden wskaźnik na jednej stacji.

In [4]:
print(con.sql('DESCRIBE SELECT * FROM stanowiska'))

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stanowiska    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ kod_stacji        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wskaznik          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wskaznik_nazwa    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ czas_usredniania  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ typ_pomiaru       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_uruchomienia │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ data_zamkniecia   │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ wojewodztwo       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ strefa    

| Kolumna | Znaczenie |
|---|---|
| `kod_stanowiska` | kod stanowiska, ten sam co w `pomiary_2024` |
| `kod_stacji` | stacja, na której stoi stanowisko |
| `wskaznik`, `wskaznik_nazwa` | kod i pełna nazwa wskaźnika, np. `PM2.5` / `pył zawieszony PM2.5` |
| `czas_usredniania` | zapisany słownie: `1-godzinny`, `24-godzinny`, `2-tygodniowy`, `miesięczny` |
| `typ_pomiaru` | `automatyczny` (analizator, wynik na bieżąco) albo `manualny` (próbka pobrana na filtr i zważona lub zbadana w laboratorium) |
| `data_uruchomienia`, `data_zamkniecia` | okres działania stanowiska |
| `wojewodztwo` | województwo |
| `strefa` | strefa oceny jakości powietrza, np. aglomeracja albo pozostały obszar województwa |

Klucz: `kod_stanowiska`.

Czas uśredniania jest zapisany inaczej niż w `pomiary_2024` (`1-godzinny` zamiast `1g`). Dlatego pomiary łączymy ze stanowiskami po `kod_stanowiska`, a nie po parze wskaźnik i czas uśredniania.

### 2.4. `openmeteo_2024`

1 wiersz = pogoda dla 1 stacji mierzącej PM2.5 w 1 godzinie.

In [5]:
print(con.sql('DESCRIBE SELECT * FROM openmeteo_2024'))

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stacji           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_czas_utc        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ temperature_2m       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ relative_humidity_2m │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ precipitation        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ surface_pressure     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ wind_speed_10m       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ wind_direction_10m   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ siatka_szerokosc     │ DOUBLE      │ YES     │ NUL

| Kolumna | Znaczenie |
|---|---|
| `kod_stacji` | kod stacji GIOŚ, wzięty z nazwy pliku JSON |
| `data_czas_utc` | godzina w UTC |
| `temperature_2m` | temperatura powietrza 2 m nad ziemią, °C |
| `relative_humidity_2m` | wilgotność względna 2 m nad ziemią, % |
| `precipitation` | suma opadu z poprzedniej godziny, mm |
| `surface_pressure` | ciśnienie na poziomie terenu (nie przeliczone do poziomu morza), hPa |
| `wind_speed_10m` | prędkość wiatru 10 m nad ziemią, km/h |
| `wind_direction_10m` | kierunek, z którego wieje wiatr, ° (0 = północ) |
| `siatka_szerokosc`, `siatka_dlugosc` | punkt siatki modelu pogodowego najbliższy stacji; to nie są dokładne współrzędne stacji |
| `wysokosc_m` | wysokość terenu n.p.m. przyjęta przez Open-Meteo dla tego punktu |
| `plik_zrodlowy` | plik JSON z warstwy RAW |

Klucz: `(kod_stacji, data_czas_utc)`.

To dane z modelu meteorologicznego (reanaliza), a nie pomiar na stacji GIOŚ.

### 2.5. Relacje między tabelami

```text
pomiary_2024   ──(kod_stanowiska)──►  stanowiska  ──(kod_stacji)──►  stacje
openmeteo_2024 ─────────────────────(kod_stacji)──────────────────►  stacje
pomiary_2024   ◄──(kod_stacji + czas)──►  openmeteo_2024     [Milestone 2]
```

- `pomiary_2024.kod_stanowiska` → `stanowiska.kod_stanowiska`
- `pomiary_2024.kod_stacji`, `stanowiska.kod_stacji`, `openmeteo_2024.kod_stacji` → `stacje.kod_stacji`
- pomiary z pogodą łączą się po stacji i czasie. GIOŚ zapisuje czas bez strefy, a Open-Meteo w UTC, więc zgodność czasu sprawdzamy w profilowaniu przed złączeniem w warstwie silver (Milestone 2).

## 3. Profil pomiarów (`pomiary_2024`)

Wszystkie statystyki liczone zapytaniami SQL bezpośrednio na pliku Parquet.

### 3.1. Liczba rekordów

In [6]:
con.sql('''
SELECT
    count(*) AS rekordy,
    count(DISTINCT kod_stacji) AS stacje,
    count(DISTINCT kod_stanowiska) AS stanowiska,
    count(DISTINCT wskaznik) AS wskazniki,
    count(DISTINCT (wskaznik, czas_usredniania)) AS wskazniki_z_czasem,
    count(DISTINCT plik_zrodlowy) AS pliki
FROM pomiary_2024
''').show(max_rows=40, max_width=250)

┌─────────┬────────┬────────────┬───────────┬────────────────────┬───────┐
│ rekordy │ stacje │ stanowiska │ wskazniki │ wskazniki_z_czasem │ pliki │
│  int64  │ int64  │   int64    │   int64   │       int64        │ int64 │
├─────────┼────────┼────────────┼───────────┼────────────────────┼───────┤
│ 8952726 │    286 │       1759 │        60 │                 64 │    28 │
└─────────┴────────┴────────────┴───────────┴────────────────────┴───────┘



Rozbicie na wskaźnik i czas uśredniania. `rekordy` to wszystkie wiersze, `pomiary` - wiersze z wartością, `braki` - wiersze z `NULL` w `wartosc`, czyli okresy bez pomiaru.

In [7]:
con.sql('''
SELECT
    wskaznik,
    czas_usredniania,
    jednostka,
    count(DISTINCT kod_stanowiska) AS stanowiska,
    count(*) AS rekordy,
    count(wartosc) AS pomiary,
    count(*) - count(wartosc) AS braki,
    round(100 * count(wartosc) / count(*), 1) AS kompletnosc_proc
FROM pomiary_2024
GROUP BY ALL
ORDER BY rekordy DESC
''').show(max_rows=70, max_width=250)

┌────────────────────┬──────────────────┬───────────┬────────────┬─────────┬─────────┬───────┬──────────────────┐
│      wskaznik      │ czas_usredniania │ jednostka │ stanowiska │ rekordy │ pomiary │ braki │ kompletnosc_proc │
│      varchar       │     varchar      │  varchar  │   int64    │  int64  │  int64  │ int64 │      double      │
├────────────────────┼──────────────────┼───────────┼────────────┼─────────┼─────────┼───────┼──────────────────┤
│ PM10               │ 1g               │ ug/m3     │        179 │ 1572336 │ 1519684 │ 52652 │             96.7 │
│ NO                 │ 1g               │ ug/m3     │        134 │ 1177056 │ 1135235 │ 41821 │             96.4 │
│ NO2                │ 1g               │ ug/m3     │        134 │ 1177056 │ 1135319 │ 41737 │             96.5 │
│ NOx                │ 1g               │ ug/m3     │        115 │ 1010160 │  973833 │ 36327 │             96.4 │
│ O3                 │ 1g               │ ug/m3     │         98 │  860832 │  839875 │ 2

Wskaźników jest 60, a par wskaźnik + czas uśredniania 64, bo NO2, SO2, PM10 i PM2.5 są mierzone zarówno godzinowo, jak i dobowo. Plik `2024_PrekursoryZielonka_1g.xlsx` zawiera 29 różnych lotnych związków organicznych z jednej stacji (Bory Tucholskie), a `2024_Jony_PM25_24g.xlsx` - 10 składników pyłu PM2.5.

### 3.2. Braki danych

Liczba `NULL` w każdej kolumnie. Braki w kluczach albo w czasie oznaczałyby błąd konwersji do bronze.

In [8]:
con.sql('''
SELECT
    count(*) - count(kod_stanowiska) AS kod_stanowiska,
    count(*) - count(kod_stacji) AS kod_stacji,
    count(*) - count(wskaznik) AS wskaznik,
    count(*) - count(czas_usredniania) AS czas_usredniania,
    count(*) - count(jednostka) AS jednostka,
    count(*) - count(data_czas) AS data_czas,
    count(*) - count(wartosc) AS wartosc,
    count(*) - count(plik_zrodlowy) AS plik_zrodlowy
FROM pomiary_2024
''').show(max_rows=40, max_width=250)

┌────────────────┬────────────┬──────────┬──────────────────┬───────────┬───────────┬─────────┬───────────────┐
│ kod_stanowiska │ kod_stacji │ wskaznik │ czas_usredniania │ jednostka │ data_czas │ wartosc │ plik_zrodlowy │
│     int64      │   int64    │  int64   │      int64       │   int64   │   int64   │  int64  │     int64     │
├────────────────┼────────────┼──────────┼──────────────────┼───────────┼───────────┼─────────┼───────────────┤
│              0 │          0 │        0 │                0 │         0 │         0 │  470793 │             0 │
└────────────────┴────────────┴──────────┴──────────────────┴───────────┴───────────┴─────────┴───────────────┘



Braki występują tylko w `wartosc`. Kompletność dla wskaźników jest w tabeli w 3.1.

Dla PM2.5 (zmienna docelowa modelu) sprawdzamy kompletność dokładniej: na stanowisko i na miesiąc. Próg 90% to minimalna kompletność wymagana dla pomiarów stałych w dyrektywie UE 2008/50/WE.

In [9]:
con.sql('''
WITH kompletnosc AS (
    SELECT
        kod_stanowiska,
        100 * count(wartosc) / count(*) AS proc
    FROM pomiary_2024
    WHERE wskaznik = 'PM2.5' AND czas_usredniania = '1g'
    GROUP BY kod_stanowiska
)
SELECT
    count(*) AS stanowiska,
    round(min(proc), 1) AS min_proc,
    round(median(proc), 1) AS mediana_proc,
    count(*) FILTER (WHERE proc < 90) AS ponizej_90_proc,
    count(*) FILTER (WHERE proc < 75) AS ponizej_75_proc
FROM kompletnosc
''').show(max_rows=40, max_width=250)

┌────────────┬──────────┬──────────────┬─────────────────┬─────────────────┐
│ stanowiska │ min_proc │ mediana_proc │ ponizej_90_proc │ ponizej_75_proc │
│   int64    │  double  │    double    │      int64      │      int64      │
├────────────┼──────────┼──────────────┼─────────────────┼─────────────────┤
│         96 │     42.7 │         98.6 │               3 │               1 │
└────────────┴──────────┴──────────────┴─────────────────┴─────────────────┘



In [10]:
con.sql('''
SELECT
    kod_stanowiska,
    count(*) - count(wartosc) AS braki,
    round(100 * count(wartosc) / count(*), 1) AS kompletnosc_proc
FROM pomiary_2024
WHERE wskaznik = 'PM2.5' AND czas_usredniania = '1g'
GROUP BY kod_stanowiska
ORDER BY kompletnosc_proc
LIMIT 10
''').show(max_rows=40, max_width=250)

┌──────────────────────────┬───────┬──────────────────┐
│      kod_stanowiska      │ braki │ kompletnosc_proc │
│         varchar          │ int64 │      double      │
├──────────────────────────┼───────┼──────────────────┤
│ MzSiedKonars-PM2.5-1g    │  5034 │             42.7 │
│ PdBialUpalna-PM2.5-1g    │   953 │             89.2 │
│ LdRadomsRoln-PM2.5-1g    │   901 │             89.7 │
│ SkPolaRuszcz-PM2.5-1g    │   874 │             90.1 │
│ KpBydPlPozna-PM2.5-1g    │   830 │             90.6 │
│ PkJasloSikor-PM2.5-1g    │   744 │             91.5 │
│ KpWloclOkrze-PM2.5-1g    │   686 │             92.2 │
│ PdGrajewoWPoMOB-PM2.5-1g │   678 │             92.3 │
│ PkMielBierna-PM2.5-1g    │   646 │             92.6 │
│ WmDzialPolnaMOB-PM2.5-1g │   588 │             93.3 │
└──────────────────────────┴───────┴──────────────────┘
  10 rows                                   3 columns



Kompletność PM2.5 na miesiąc. Od czasu odejmujemy godzinę, bo GIOŚ oznacza pomiar godziną końca okresu (pomiar z `2025-01-01 00:00` dotyczy ostatniej godziny grudnia - patrz 3.6).

In [11]:
con.sql('''
SELECT
    month(data_czas - INTERVAL 1 HOUR) AS miesiac,
    count(*) - count(wartosc) AS braki,
    round(100 * count(wartosc) / count(*), 1) AS kompletnosc_proc
FROM pomiary_2024
WHERE wskaznik = 'PM2.5' AND czas_usredniania = '1g'
GROUP BY miesiac
ORDER BY miesiac
''').show(max_rows=40, max_width=250)

┌─────────┬───────┬──────────────────┐
│ miesiac │ braki │ kompletnosc_proc │
│  int64  │ int64 │      double      │
├─────────┼───────┼──────────────────┤
│       1 │  2514 │             96.5 │
│       2 │   761 │             98.9 │
│       3 │   702 │             99.0 │
│       4 │   927 │             98.7 │
│       5 │  2486 │             96.5 │
│       6 │  2131 │             96.9 │
│       7 │  2260 │             96.8 │
│       8 │  2651 │             96.3 │
│       9 │  2723 │             96.1 │
│      10 │  1892 │             97.4 │
│      11 │  1795 │             97.4 │
│      12 │  4774 │             93.3 │
└─────────┴───────┴──────────────────┘
  12 rows                  3 columns



### 3.3. Statystyki wartości na wskaźnik

Min, max, średnia i percentyle liczone osobno dla każdego wskaźnika, bo jednostki i typowe poziomy są różne. `max_do_p99_9` to stosunek maksimum do percentyla 99,9: duża wartość oznacza pojedyncze wartości daleko od reszty rozkładu.

In [12]:
con.sql('''
SELECT
    wskaznik,
    czas_usredniania,
    jednostka,
    min(wartosc) AS min,
    round(quantile_cont(wartosc, 0.01), 2) AS p1,
    round(median(wartosc), 2) AS p50,
    round(avg(wartosc), 2) AS srednia,
    round(quantile_cont(wartosc, 0.99), 2) AS p99,
    round(quantile_cont(wartosc, 0.999), 2) AS p99_9,
    max(wartosc) AS max,
    round(max(wartosc) / nullif(quantile_cont(wartosc, 0.999), 0), 1) AS max_do_p99_9
FROM pomiary_2024
GROUP BY ALL
ORDER BY wskaznik, czas_usredniania
''').show(max_rows=70, max_width=250)

┌────────────────────┬──────────────────┬───────────┬────────┬────────┬────────┬─────────┬────────┬────────┬─────────┬──────────────┐
│      wskaznik      │ czas_usredniania │ jednostka │  min   │   p1   │  p50   │ srednia │  p99   │ p99_9  │   max   │ max_do_p99_9 │
│      varchar       │     varchar      │  varchar  │ double │ double │ double │ double  │ double │ double │ double  │    double    │
├────────────────────┼──────────────────┼───────────┼────────┼────────┼────────┼─────────┼────────┼────────┼─────────┼──────────────┤
│ 123trimetylobenzen │ 1g               │ ug/m3     │    0.0 │   0.01 │   0.14 │    0.19 │   0.83 │   1.14 │    1.25 │          1.1 │
│ 124trimetylobenzen │ 1g               │ ug/m3     │    0.0 │   0.02 │   1.32 │    1.24 │   2.73 │   2.89 │    2.91 │          1.0 │
│ 135trimetylobenzen │ 1g               │ ug/m3     │    0.0 │   0.04 │    0.4 │    0.52 │   2.24 │   3.75 │    3.96 │          1.1 │
│ 13butadien         │ 1g               │ ug/m3     │    0.0 │

### 3.4. Podejrzane wartości

**Wartości ujemne i zera.** Stężenie nie może być ujemne. Zero dokładnie `0.0` jest możliwe dla niektórych gazów przy bardzo czystym powietrzu, ale dla pyłu oznacza raczej wartość poniżej progu wykrywalności przyrządu albo jego awarię.

In [13]:
con.sql('''
SELECT
    wskaznik,
    czas_usredniania,
    count(wartosc) AS pomiary,
    count(*) FILTER (WHERE wartosc < 0) AS ujemne,
    count(*) FILTER (WHERE wartosc = 0) AS zera,
    round(100 * count(*) FILTER (WHERE wartosc = 0) / count(wartosc), 2) AS zera_proc
FROM pomiary_2024
GROUP BY ALL
HAVING ujemne > 0 OR zera > 0
ORDER BY zera DESC
''').show(max_rows=70, max_width=250)

┌────────────────────┬──────────────────┬─────────┬────────┬───────┬───────────┐
│      wskaznik      │ czas_usredniania │ pomiary │ ujemne │ zera  │ zera_proc │
│      varchar       │     varchar      │  int64  │ int64  │ int64 │  double   │
├────────────────────┼──────────────────┼─────────┼────────┼───────┼───────────┤
│ NO                 │ 1g               │ 1135235 │      0 │ 14856 │      1.31 │
│ SO2                │ 1g               │  665147 │      0 │  6489 │      0.98 │
│ C6H6               │ 1g               │  535548 │      0 │  2382 │      0.44 │
│ PM10               │ 1g               │ 1519684 │      0 │   990 │      0.07 │
│ PM2.5              │ 1g               │  817648 │      0 │   728 │      0.09 │
│ O3                 │ 1g               │  839875 │      0 │   107 │      0.01 │
│ Ni(PM10)           │ 24g              │   20431 │      0 │   105 │      0.51 │
│ NO2                │ 1g               │ 1135319 │      0 │    64 │      0.01 │
│ oksylen            │ 1g   

In [14]:
con.sql('''
SELECT kod_stanowiska, data_czas, wartosc, jednostka
FROM pomiary_2024
WHERE wartosc < 0
''').show(max_rows=40, max_width=250)

┌────────────────────────────┬─────────────────────┬─────────┬───────────┐
│       kod_stanowiska       │      data_czas      │ wartosc │ jednostka │
│          varchar           │      timestamp      │ double  │  varchar  │
├────────────────────────────┼─────────────────────┼─────────┼───────────┤
│ WmPuszczaBor-OC(PM2.5)-24g │ 2024-12-16 00:00:00 │   -0.35 │ ug/m3     │
│ WmPuszczaBor-OC(PM2.5)-24g │ 2024-12-17 00:00:00 │   -0.37 │ ug/m3     │
└────────────────────────────┴─────────────────────┴─────────┴───────────┘



Zera w PM2.5 i PM10 nie są rozłożone równo - skupiają się na kilku stanowiskach:

In [15]:
con.sql('''
SELECT
    kod_stanowiska,
    count(*) FILTER (WHERE wartosc = 0) AS zera,
    count(wartosc) AS pomiary
FROM pomiary_2024
WHERE wskaznik IN ('PM2.5', 'PM10') AND czas_usredniania = '1g'
GROUP BY kod_stanowiska
HAVING zera > 0
ORDER BY zera DESC
LIMIT 8
''').show(max_rows=40, max_width=250)

┌───────────────────────┬───────┬─────────┐
│    kod_stanowiska     │ zera  │ pomiary │
│        varchar        │ int64 │  int64  │
├───────────────────────┼───────┼─────────┤
│ PdBorsukowiz-PM2.5-1g │   427 │    8681 │
│ PdBorsukowiz-PM10-1g  │   306 │    8681 │
│ PdSuwPulask2-PM2.5-1g │   198 │    8473 │
│ DsDziePilsud-PM10-1g  │   185 │    8605 │
│ PdBialWarsza-PM10-1g  │   180 │    8627 │
│ DsTrzebniMaj-PM10-1g  │   139 │    8690 │
│ PdBialUpalna-PM2.5-1g │    91 │    7831 │
│ DsLegAlRzecz-PM10-1g  │    71 │    7780 │
└───────────────────────┴───────┴─────────┘



**Spójność fizyczna między wskaźnikami.** Zamiast arbitralnych progów sprawdzamy zależności, które muszą zachodzić z definicji:

- PM2.5 to część PM10 (pył o średnicy do 2,5 µm zawiera się w pyle do 10 µm), więc na tej samej stacji w tej samej godzinie PM2.5 ≤ PM10;
- NOx to suma NO i NO2 (przeliczona na NO2), więc NO2 ≤ NOx.

Każda stacja ma najwyżej jedno godzinowe stanowisko danego wskaźnika, więc porównanie w obrębie stacji i godziny jest jednoznaczne.

In [16]:
con.sql('''
WITH pyl AS (
    SELECT
        kod_stacji,
        data_czas,
        max(wartosc) FILTER (WHERE wskaznik = 'PM2.5') AS pm25,
        max(wartosc) FILTER (WHERE wskaznik = 'PM10') AS pm10
    FROM pomiary_2024
    WHERE czas_usredniania = '1g' AND wskaznik IN ('PM2.5', 'PM10')
    GROUP BY kod_stacji, data_czas
)
SELECT
    count(*) FILTER (WHERE pm25 IS NOT NULL AND pm10 IS NOT NULL) AS godziny_z_oboma,
    count(*) FILTER (WHERE pm25 > pm10) AS pm25_wieksze_od_pm10,
    count(DISTINCT kod_stacji) FILTER (WHERE pm25 > pm10) AS stacje,
    round(max(pm25 - pm10), 1) AS max_nadwyzka
FROM pyl
''').show(max_rows=40, max_width=250)

┌─────────────────┬──────────────────────┬────────┬──────────────┐
│ godziny_z_oboma │ pm25_wieksze_od_pm10 │ stacje │ max_nadwyzka │
│      int64      │        int64         │ int64  │    double    │
├─────────────────┼──────────────────────┼────────┼──────────────┤
│          773061 │                 1003 │      8 │          6.4 │
└─────────────────┴──────────────────────┴────────┴──────────────┘



In [17]:
con.sql('''
WITH tlenki AS (
    SELECT
        kod_stacji,
        data_czas,
        max(wartosc) FILTER (WHERE wskaznik = 'NO2') AS no2,
        max(wartosc) FILTER (WHERE wskaznik = 'NOx') AS nox
    FROM pomiary_2024
    WHERE czas_usredniania = '1g' AND wskaznik IN ('NO2', 'NOx')
    GROUP BY kod_stacji, data_czas
)
SELECT
    count(*) FILTER (WHERE no2 IS NOT NULL AND nox IS NOT NULL) AS godziny_z_oboma,
    count(*) FILTER (WHERE no2 > nox) AS no2_wieksze_od_nox,
    count(*) FILTER (WHERE no2 > nox + 5) AS wieksze_o_ponad_5,
    count(DISTINCT kod_stacji) FILTER (WHERE no2 > nox) AS stacje,
    round(max(no2 - nox), 1) AS max_nadwyzka
FROM tlenki
''').show(max_rows=40, max_width=250)

┌─────────────────┬────────────────────┬───────────────────┬────────┬──────────────┐
│ godziny_z_oboma │ no2_wieksze_od_nox │ wieksze_o_ponad_5 │ stacje │ max_nadwyzka │
│      int64      │       int64        │       int64       │ int64  │    double    │
├─────────────────┼────────────────────┼───────────────────┼────────┼──────────────┤
│          973788 │               5220 │                10 │     34 │          9.9 │
└─────────────────┴────────────────────┴───────────────────┴────────┴──────────────┘



PM2.5 przekracza PM10 w 1003 godzinach na 8 stacjach, ale najwyżej o 6,4 µg/m³. NO2 przekracza NOx w 5220 godzinach na 34 stacjach, z czego tylko 10 razy o więcej niż 5 µg/m³ (maksymalnie 9,9). Oba wskaźniki mierzą osobne przyrządy, więc to drobne niezgodności na poziomie kilku µg/m³, a nie wartości wielokrotnie przekraczające fizyczne ograniczenie.

**Wartości skrajne.** Liczba wartości powyżej p99,9 wynosi z definicji około 0,1% pomiarów, więc sama w sobie nic nie mówi. Zamiast niej pokazujemy największe wartości z datą - pozwala to odróżnić pojedynczy odczyt od epizodu trwającego kilka godzin albo obejmującego kilka stacji naraz.

In [18]:
con.sql('''
SELECT wskaznik, kod_stanowiska, data_czas, wartosc, jednostka
FROM pomiary_2024
WHERE czas_usredniania = '1g'
    AND wskaznik IN ('PM2.5', 'PM10', 'SO2', 'NOx')
    AND wartosc IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY wskaznik ORDER BY wartosc DESC) <= 5
ORDER BY wskaznik, wartosc DESC
''').show(max_rows=40, max_width=250)

┌──────────┬───────────────────────┬─────────────────────┬─────────┬───────────┐
│ wskaznik │    kod_stanowiska     │      data_czas      │ wartosc │ jednostka │
│ varchar  │        varchar        │      timestamp      │ double  │  varchar  │
├──────────┼───────────────────────┼─────────────────────┼─────────┼───────────┤
│ NOx      │ SlCzestoArmK-NOx-1g   │ 2024-11-06 18:00:00 │  1104.5 │ ug/m3     │
│ NOx      │ DsWrocAlWisn-NOx-1g   │ 2024-12-02 08:00:00 │  1036.4 │ ug/m3     │
│ NOx      │ SlCzestoArmK-NOx-1g   │ 2024-11-06 19:00:00 │  1031.9 │ ug/m3     │
│ NOx      │ KpWloclOkrze-NOx-1g   │ 2024-11-05 18:00:00 │   984.1 │ ug/m3     │
│ NOx      │ DsWrocAlWisn-NOx-1g   │ 2024-12-02 09:00:00 │   964.0 │ ug/m3     │
│ PM10     │ WpMosinaCzer-PM10-1g  │ 2024-01-10 21:00:00 │   526.5 │ ug/m3     │
│ PM10     │ WpMosinaCzer-PM10-1g  │ 2024-01-10 22:00:00 │   513.4 │ ug/m3     │
│ PM10     │ SlRybniBorki-PM10-1g  │ 2024-01-10 22:00:00 │   472.2 │ ug/m3     │
│ PM10     │ PdAugustowUz-PM

Najwyższe wartości nie są pojedynczymi odczytami: PM2.5 i PM10 osiągają maksima w kilku stacjach jednocześnie w dniach 8-10 stycznia, a skrajne SO2 (1353,8 µg/m³, 28 razy więcej niż p99,9) występuje w tych samych godzinach 27 lutego na dwóch stacjach w Szczecinie. To wzorzec epizodu obejmującego obszar, a nie awarii jednego przyrządu.

### 3.5. Duplikaty

Klucz `(kod_stanowiska, data_czas)` powinien być unikalny.

In [19]:
con.sql('''
SELECT
    count(*) AS rekordy,
    count(DISTINCT (kod_stanowiska, data_czas)) AS unikalne_klucze,
    count(*) - count(DISTINCT (kod_stanowiska, data_czas)) AS nadmiarowe_wiersze
FROM pomiary_2024
''').show(max_rows=40, max_width=250)

┌─────────┬─────────────────┬────────────────────┐
│ rekordy │ unikalne_klucze │ nadmiarowe_wiersze │
│  int64  │      int64      │       int64        │
├─────────┼─────────────────┼────────────────────┤
│ 8952726 │         8943942 │               8784 │
└─────────┴─────────────────┴────────────────────┘



In [20]:
con.sql('''
WITH dubel AS (
    SELECT
        kod_stanowiska,
        data_czas,
        list(plik_zrodlowy ORDER BY plik_zrodlowy) AS pliki,
        min(wartosc) IS DISTINCT FROM max(wartosc)
            OR count(wartosc) NOT IN (0, count(*)) AS rozne_wartosci
    FROM pomiary_2024
    GROUP BY kod_stanowiska, data_czas
    HAVING count(*) > 1
)
SELECT
    kod_stanowiska,
    pliki,
    count(*) AS zdublowane_okresy,
    count(*) FILTER (WHERE rozne_wartosci) AS z_roznymi_wartosciami
FROM dubel
GROUP BY ALL
''').show(max_rows=40, max_width=250)

┌──────────────────────┬──────────────────────────────────────────────────────┬───────────────────┬───────────────────────┐
│    kod_stanowiska    │                        pliki                         │ zdublowane_okresy │ z_roznymi_wartosciami │
│       varchar        │                      varchar[]                       │       int64       │         int64         │
├──────────────────────┼──────────────────────────────────────────────────────┼───────────────────┼───────────────────────┤
│ KpZielBoryTu-C6H6-1g │ [2024_C6H6_1g.xlsx, 2024_PrekursoryZielonka_1g.xlsx] │              8784 │                     0 │
└──────────────────────┴──────────────────────────────────────────────────────┴───────────────────┴───────────────────────┘



Wszystkie duplikaty to jedno stanowisko, które występuje w dwóch plikach z identycznymi wartościami.

Do analiz szeregów czasowych (3.6-3.8) potrzebujemy jednego wiersza na stanowisko i godzinę, inaczej `lag`/`lead` porównywałyby wiersz z jego kopią. Zakładamy więc widok bez duplikatów. To tylko widok na potrzeby profilowania - dane w bronze zostają bez zmian.

In [21]:
con.sql('''
CREATE OR REPLACE VIEW pomiary_bez_dubli AS
SELECT *
FROM pomiary_2024
QUALIFY row_number() OVER (PARTITION BY kod_stanowiska, data_czas ORDER BY plik_zrodlowy) = 1
''')

con.sql('SELECT count(*) AS rekordy FROM pomiary_bez_dubli').show()

┌─────────┐
│ rekordy │
│  int64  │
├─────────┤
│ 8943942 │
└─────────┘



### 3.6. Czas

Zakres dat, wpisy spoza 2024 r., wpisy dobowe niebędące północą i znaczniki niebędące pełną godziną.

In [22]:
con.sql('''
SELECT
    czas_usredniania,
    min(data_czas) AS pierwszy,
    max(data_czas) AS ostatni,
    count(*) FILTER (WHERE year(data_czas) <> 2024) AS poza_2024,
    count(*) FILTER (WHERE data_czas = TIMESTAMP '2024-01-01 00:00:00') AS o_polnocy_1_stycznia,
    count(*) FILTER (WHERE czas_usredniania = '24g' AND data_czas::TIME <> TIME '00:00:00') AS dobowe_nie_o_polnocy,
    count(*) FILTER (WHERE minute(data_czas) <> 0 OR second(data_czas) <> 0) AS nie_pelna_godzina
FROM pomiary_2024
GROUP BY czas_usredniania
''').show(max_rows=40, max_width=250)

┌──────────────────┬─────────────────────┬─────────────────────┬───────────┬──────────────────────┬──────────────────────┬───────────────────┐
│ czas_usredniania │      pierwszy       │       ostatni       │ poza_2024 │ o_polnocy_1_stycznia │ dobowe_nie_o_polnocy │ nie_pelna_godzina │
│     varchar      │      timestamp      │      timestamp      │   int64   │        int64         │        int64         │       int64       │
├──────────────────┼─────────────────────┼─────────────────────┼───────────┼──────────────────────┼──────────────────────┼───────────────────┤
│ 24g              │ 2024-01-01 00:00:00 │ 2024-12-31 00:00:00 │         0 │                  773 │                    0 │                 0 │
│ 1g               │ 2024-01-01 01:00:00 │ 2025-01-01 00:00:00 │       987 │                    0 │                    0 │                 0 │
└──────────────────┴─────────────────────┴─────────────────────┴───────────┴──────────────────────┴──────────────────────┴───────────────────┘

Pomiary godzinowe zaczynają się o `2024-01-01 01:00`, a kończą o `2025-01-01 00:00`. Wpisy z 2025 r. (987 = jeden na każde z 986 stanowisk godzinowych plus kopia zdublowanego stanowiska z 3.5) nie są więc błędem: GIOŚ oznacza pomiar godziną końca okresu uśredniania, czyli `01:00` to średnia z 00:00-01:00. Trzeba to uwzględnić przy łączeniu z pogodą, w której znacznik oznacza początek godziny.

Czy każde stanowisko ma pełną siatkę czasu (8784 godziny w roku przestępnym albo 366 dób)?

In [23]:
con.sql('''
WITH na_stanowisko AS (
    SELECT kod_stanowiska, czas_usredniania, count(*) AS rekordy
    FROM pomiary_bez_dubli
    GROUP BY ALL
)
SELECT czas_usredniania, rekordy AS rekordy_na_stanowisko, count(*) AS stanowiska
FROM na_stanowisko
GROUP BY ALL
ORDER BY ALL
''').show(max_rows=40, max_width=250)

┌──────────────────┬───────────────────────┬────────────┐
│ czas_usredniania │ rekordy_na_stanowisko │ stanowiska │
│     varchar      │         int64         │   int64    │
├──────────────────┼───────────────────────┼────────────┤
│ 1g               │                  8784 │        986 │
│ 24g              │                   366 │        773 │
└──────────────────┴───────────────────────┴────────────┘



Każde stanowisko ma wiersz dla każdego okresu, bo plik GIOŚ to pełna tabela godzin × stanowisk. Brakujący pomiar to więc nie brakujący wiersz, tylko `NULL` w `wartosc`.

**Luki w szeregu.** Żeby zmierzyć, jak długie są przerwy w pomiarach, dzielimy każdy szereg godzinowy na ciągi kolejnych godzin o tej samej wartości (metoda *gaps and islands*): nowy ciąg zaczyna się, gdy wartość różni się od poprzedniej godziny. Ciąg `NULL` to luka, a ciąg tej samej liczby - stała wartość (wykorzystamy go w 3.7).

Wynik zapisujemy jako tabelę tymczasową w pamięci, bo korzystają z niego dwie kolejne analizy.

In [24]:
con.sql('''
CREATE OR REPLACE TEMP TABLE ciagi AS
WITH kolejne AS (
    SELECT
        kod_stanowiska,
        wskaznik,
        data_czas,
        wartosc,
        CASE WHEN wartosc IS NOT DISTINCT FROM lag(wartosc) OVER w THEN 0 ELSE 1 END AS nowy_ciag
    FROM pomiary_bez_dubli
    WHERE czas_usredniania = '1g'
    WINDOW w AS (PARTITION BY kod_stanowiska ORDER BY data_czas)
),
numerowane AS (
    SELECT *, sum(nowy_ciag) OVER (PARTITION BY kod_stanowiska ORDER BY data_czas) AS nr_ciagu
    FROM kolejne
)
SELECT
    kod_stanowiska,
    wskaznik,
    nr_ciagu,
    any_value(wartosc) AS wartosc,
    min(data_czas) AS poczatek,
    max(data_czas) AS koniec,
    count(*) AS godziny
FROM numerowane
GROUP BY kod_stanowiska, wskaznik, nr_ciagu
''')

con.sql('SELECT count(*) AS ciagi FROM ciagi').show()

┌─────────┐
│  ciagi  │
│  int64  │
├─────────┤
│ 7774651 │
└─────────┘



Luki w głównych wskaźnikach godzinowych:

In [25]:
con.sql('''
SELECT
    wskaznik,
    count(*) AS luki,
    count(*) FILTER (WHERE godziny >= 24) AS od_doby,
    count(*) FILTER (WHERE godziny >= 24 * 7) AS od_tygodnia,
    max(godziny) AS najdluzsza_h
FROM ciagi
WHERE wartosc IS NULL AND wskaznik IN ('PM10', 'PM2.5', 'NO2', 'NO', 'NOx', 'O3', 'SO2', 'CO', 'C6H6')
GROUP BY wskaznik
ORDER BY wskaznik
''').show(max_rows=40, max_width=250)

┌──────────┬───────┬─────────┬─────────────┬──────────────┐
│ wskaznik │ luki  │ od_doby │ od_tygodnia │ najdluzsza_h │
│ varchar  │ int64 │  int64  │    int64    │    int64     │
├──────────┼───────┼─────────┼─────────────┼──────────────┤
│ C6H6     │  4374 │     127 │          22 │         2905 │
│ CO       │  2051 │      57 │           3 │         2896 │
│ NO       │  5943 │     151 │          23 │         6274 │
│ NO2      │  5893 │     151 │          23 │         6274 │
│ NOx      │  4803 │     114 │          19 │         6274 │
│ O3       │  2368 │     131 │          15 │         2896 │
│ PM10     │  4051 │     288 │          45 │         4993 │
│ PM2.5    │  2721 │     127 │          29 │         4993 │
│ SO2      │  3714 │      84 │          14 │         2905 │
└──────────┴───────┴─────────┴─────────────┴──────────────┘



In [26]:
con.sql('''
SELECT kod_stanowiska, poczatek, koniec, godziny
FROM ciagi
WHERE wartosc IS NULL AND wskaznik = 'PM2.5'
ORDER BY godziny DESC
LIMIT 8
''').show(max_rows=40, max_width=250)

┌──────────────────────────┬─────────────────────┬─────────────────────┬─────────┐
│      kod_stanowiska      │      poczatek       │       koniec        │ godziny │
│         varchar          │      timestamp      │      timestamp      │  int64  │
├──────────────────────────┼─────────────────────┼─────────────────────┼─────────┤
│ MzSiedKonars-PM2.5-1g    │ 2024-06-07 00:00:00 │ 2025-01-01 00:00:00 │    4993 │
│ PdBialUpalna-PM2.5-1g    │ 2024-01-01 01:00:00 │ 2024-01-25 11:00:00 │     587 │
│ PdGrajewoWPoMOB-PM2.5-1g │ 2024-12-09 12:00:00 │ 2025-01-01 00:00:00 │     541 │
│ KpWloclOkrze-PM2.5-1g    │ 2024-08-13 02:00:00 │ 2024-09-03 07:00:00 │     510 │
│ MzLegZegrzyn-PM2.5-1g    │ 2024-11-29 12:00:00 │ 2024-12-20 09:00:00 │     502 │
│ LdLodzCzerni-PM2.5-1g    │ 2024-04-28 11:00:00 │ 2024-05-18 21:00:00 │     491 │
│ SkPolaRuszcz-PM2.5-1g    │ 2024-09-11 02:00:00 │ 2024-09-30 20:00:00 │     475 │
│ PkRzeszSloci-PM2.5-1g    │ 2024-12-12 09:00:00 │ 2025-01-01 00:00:00 │     472 │
└───

Najdłuższa luka PM2.5 (`MzSiedKonars`, od 7 czerwca do końca roku) pokrywa się z datą zamknięcia stanowiska w metadanych (2024-06-06). Pozostałe długie luki trwają po 2-3 tygodnie.

### 3.7. Zawieszony czujnik

Przyrząd, który się zawiesił, zwraca przez wiele godzin tę samą liczbę. Stężenia pyłu zmieniają się z godziny na godzinę (mediana zmiany PM2.5 to ok. 1 µg/m³ - patrz 3.8), więc kilka godzin identycznej wartości jest podejrzane. Nie ma jednego obiektywnego progu, dlatego pokazujemy liczbę ciągów od 6, 12 i 24 godzin, osobno dla zer i dla innych wartości.

In [27]:
con.sql('''
SELECT
    wskaznik,
    wartosc = 0 AS zero,
    count(*) FILTER (WHERE godziny >= 6) AS od_6h,
    count(*) FILTER (WHERE godziny >= 12) AS od_12h,
    count(*) FILTER (WHERE godziny >= 24) AS od_24h,
    count(DISTINCT kod_stanowiska) AS stanowiska,
    max(godziny) AS najdluzszy_h
FROM ciagi
WHERE wartosc IS NOT NULL AND godziny >= 6 AND wskaznik IN ('PM10', 'PM2.5', 'NO2', 'NO', 'NOx', 'O3', 'SO2', 'CO', 'C6H6')
GROUP BY ALL
ORDER BY wskaznik, zero
''').show(max_rows=40, max_width=250)

┌──────────┬─────────┬───────┬────────┬────────┬────────────┬──────────────┐
│ wskaznik │  zero   │ od_6h │ od_12h │ od_24h │ stanowiska │ najdluzszy_h │
│ varchar  │ boolean │ int64 │ int64  │ int64  │   int64    │    int64     │
├──────────┼─────────┼───────┼────────┼────────┼────────────┼──────────────┤
│ C6H6     │ false   │  2346 │    361 │     50 │         43 │           73 │
│ C6H6     │ true    │    97 │     16 │      0 │          8 │           22 │
│ NO       │ false   │  2881 │    304 │      4 │         90 │           50 │
│ NO       │ true    │   428 │    134 │     35 │         26 │          120 │
│ NO2      │ false   │    22 │      0 │      0 │          7 │           10 │
│ NO2      │ true    │     1 │      0 │      0 │          1 │            6 │
│ NOx      │ false   │     2 │      1 │      1 │          2 │           25 │
│ NOx      │ true    │     1 │      0 │      0 │          1 │           11 │
│ O3       │ false   │     6 │      0 │      0 │          3 │            8 │

Dla gazów o niskich stężeniach (SO2, NO, benzen) długie ciągi stałej wartości mogą wynikać z dokładności zapisu: wartości są podawane z rozdzielczością 0,1 (SO2, NO) albo 0,01 (benzen), a typowy poziom jest bliski zera. Dla pyłu to mniej prawdopodobne. Najdłuższe ciągi dla PM:

In [28]:
con.sql('''
SELECT kod_stanowiska, wartosc, poczatek, koniec, godziny
FROM ciagi
WHERE wartosc IS NOT NULL AND wskaznik IN ('PM2.5', 'PM10') AND godziny >= 12
ORDER BY godziny DESC
''').show(max_rows=40, max_width=250)

┌──────────────────────────┬─────────┬─────────────────────┬─────────────────────┬─────────┐
│      kod_stanowiska      │ wartosc │      poczatek       │       koniec        │ godziny │
│         varchar          │ double  │      timestamp      │      timestamp      │  int64  │
├──────────────────────────┼─────────┼─────────────────────┼─────────────────────┼─────────┤
│ PdBorsukowiz-PM10-1g     │     0.0 │ 2024-02-17 18:00:00 │ 2024-02-18 23:00:00 │      30 │
│ PdBorsukowiz-PM2.5-1g    │     0.0 │ 2024-02-17 18:00:00 │ 2024-02-18 23:00:00 │      30 │
│ PdBorsukowiz-PM2.5-1g    │     0.0 │ 2024-09-30 13:00:00 │ 2024-10-01 12:00:00 │      24 │
│ PdBorsukowiz-PM2.5-1g    │     0.0 │ 2024-12-17 01:00:00 │ 2024-12-17 22:00:00 │      22 │
│ PdBorsukowiz-PM2.5-1g    │     0.0 │ 2024-11-02 00:00:00 │ 2024-11-02 21:00:00 │      22 │
│ PdBorsukowiz-PM10-1g     │     0.0 │ 2024-12-17 01:00:00 │ 2024-12-17 21:00:00 │      21 │
│ PdBorsukowiz-PM2.5-1g    │     0.0 │ 2024-09-29 10:00:00 │ 2024-09-3

Najdłuższe ciągi stałej wartości dla pyłu to prawie wyłącznie zera na stacji `PdBorsukowiz`, równocześnie na stanowisku PM2.5 i PM10 (np. 30 godzin 17-18 lutego). Ciągi niezerowe są krótsze (do 17 godzin) i dotyczą niskich wartości.

### 3.8. Skoki

Najpierw rozkład zmiany wartości między kolejnymi godzinami dla pyłu, żeby dobrać próg na podstawie danych:

In [29]:
con.sql('''
WITH zmiany AS (
    SELECT
        wskaznik,
        abs(wartosc - lag(wartosc) OVER (PARTITION BY kod_stanowiska ORDER BY data_czas)) AS zmiana
    FROM pomiary_bez_dubli
    WHERE czas_usredniania = '1g' AND wskaznik IN ('PM2.5', 'PM10')
)
SELECT
    wskaznik,
    round(median(zmiana), 1) AS p50,
    round(quantile_cont(zmiana, 0.99), 1) AS p99,
    round(quantile_cont(zmiana, 0.999), 1) AS p99_9,
    round(quantile_cont(zmiana, 0.9999), 1) AS p99_99,
    max(zmiana) AS max
FROM zmiany
GROUP BY wskaznik
''').show(max_rows=40, max_width=250)

┌──────────┬────────┬────────┬────────┬────────┬────────┐
│ wskaznik │  p50   │  p99   │ p99_9  │ p99_99 │  max   │
│ varchar  │ double │ double │ double │ double │ double │
├──────────┼────────┼────────┼────────┼────────┼────────┤
│ PM2.5    │    1.0 │   18.6 │   48.0 │   97.8 │  322.3 │
│ PM10     │    2.1 │   28.4 │   69.8 │  139.8 │  324.3 │
└──────────┴────────┴────────┴────────┴────────┴────────┘



Pojedynczy skok w górę może być początkiem prawdziwego epizodu smogowego. Za pik uznajemy więc godzinę, która jest wyższa **zarówno od poprzedniej, jak i od następnej** o więcej niż X µg/m³ - wartość wyskakuje na jedną godzinę i od razu wraca. Progi 50 i 100 µg/m³ odpowiadają mniej więcej percentylom 99,9 i 99,99 zmian godzinowych PM2.5.

In [30]:
con.sql('''
WITH sasiedzi AS (
    SELECT
        wskaznik,
        wartosc,
        wartosc - lag(wartosc) OVER w AS wzrost,
        wartosc - lead(wartosc) OVER w AS spadek
    FROM pomiary_bez_dubli
    WHERE czas_usredniania = '1g' AND wskaznik IN ('PM2.5', 'PM10')
    WINDOW w AS (PARTITION BY kod_stanowiska ORDER BY data_czas)
)
SELECT
    wskaznik,
    count(wartosc) AS pomiary,
    count(*) FILTER (WHERE wzrost > 50 AND spadek > 50) AS piki_ponad_50,
    count(*) FILTER (WHERE wzrost > 100 AND spadek > 100) AS piki_ponad_100
FROM sasiedzi
GROUP BY wskaznik
''').show(max_rows=40, max_width=250)

┌──────────┬─────────┬───────────────┬────────────────┐
│ wskaznik │ pomiary │ piki_ponad_50 │ piki_ponad_100 │
│ varchar  │  int64  │     int64     │     int64      │
├──────────┼─────────┼───────────────┼────────────────┤
│ PM10     │ 1519684 │           530 │             83 │
│ PM2.5    │  817648 │            91 │             14 │
└──────────┴─────────┴───────────────┴────────────────┘



In [31]:
con.sql('''
WITH sasiedzi AS (
    SELECT
        kod_stanowiska,
        data_czas,
        lag(wartosc) OVER w AS poprzednia,
        wartosc,
        lead(wartosc) OVER w AS nastepna
    FROM pomiary_bez_dubli
    WHERE czas_usredniania = '1g' AND wskaznik = 'PM2.5'
    WINDOW w AS (PARTITION BY kod_stanowiska ORDER BY data_czas)
)
SELECT *
FROM sasiedzi
WHERE wartosc - poprzednia > 50 AND wartosc - nastepna > 50
ORDER BY wartosc - greatest(poprzednia, nastepna) DESC
LIMIT 10
''').show(max_rows=40, max_width=250)

┌──────────────────────────┬─────────────────────┬────────────┬─────────┬──────────┐
│      kod_stanowiska      │      data_czas      │ poprzednia │ wartosc │ nastepna │
│         varchar          │      timestamp      │   double   │ double  │  double  │
├──────────────────────────┼─────────────────────┼────────────┼─────────┼──────────┤
│ WmDzialPolnaMOB-PM2.5-1g │ 2024-03-09 18:00:00 │       42.7 │   365.0 │     67.0 │
│ DsWrocAlWisn-PM2.5-1g    │ 2024-03-31 15:00:00 │       46.5 │   260.4 │     65.1 │
│ SkStaraZlota-PM2.5-1g    │ 2024-01-10 11:00:00 │      134.0 │   296.9 │     37.6 │
│ WmDzialPolnaMOB-PM2.5-1g │ 2024-03-25 21:00:00 │       96.0 │   268.4 │    126.7 │
│ WmDzialPolnaMOB-PM2.5-1g │ 2024-07-23 11:00:00 │        4.5 │   141.5 │      2.7 │
│ DsKudowaSzkoMOB-PM2.5-1g │ 2024-11-07 21:00:00 │       89.9 │   220.3 │     43.4 │
│ WmDzialPolnaMOB-PM2.5-1g │ 2024-07-22 19:00:00 │       14.8 │   177.8 │     54.6 │
│ SkSkarz1MajaMOB-PM2.5-1g │ 2024-01-16 18:00:00 │       47.9 │  

Piki PM2.5 powtarzają się na tych samych stanowiskach, najczęściej na `WmDzialPolnaMOB` (stacja mobilna): 19 z 91 pików ponad 50 µg/m³. Skrajny przypadek: 42,7 → 365,0 → 67,0 µg/m³ w ciągu trzech godzin.